# 75. 배포용 원판 추적기와 코사인 후보 비교

Mac 단독, Mac이 비었을 때 CutMix로 보완하는 모드, 코사인 후보 단독을 같은 검증 원본에서 평가합니다. 각 모드·영상의 첫 프레임에서 궤적을 새로 시작합니다. 입력 640과 배포 경계 규칙을 유지했습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
folder = ROOT / "outputs/plate_deployment_comparison/20261126"
summary = pd.read_csv(folder / "summary.csv").set_index("mode")
athletes = pd.read_csv(folder / "athletes.csv")
frames = pd.read_csv(folder / "frames.csv", dtype={"match":"boolean"})
timing = pd.read_csv(folder / "timing.csv")
display(summary.round(2))
display(timing.groupby("mode")[["frames","seconds"]].sum().round(2))
display(pd.read_csv(folder / "false_positives.csv")[["case","phase","scored","fp_frames"]])

## 어느 선수의 궤적이 끊기고, 상승량이 달라졌나

gap_indices에는 좌표가 없던 원본 프레임 번호가 들어 있습니다. 연속 누락의 최장 길이는 longest_gap입니다. 허브 상승량은 무릎 통과→접촉으로 측정합니다. 아래 그래프의 양수는 수동값보다 크게 측정한 경우입니다.

In [ ]:
display(athletes[["mode","video_id","misses","false_positives","gap_indices","longest_gap","manual_rise_px","rise_error_px"]].round(2))
errors = athletes.pivot(index="video_id", columns="mode", values="rise_error_px")[summary.index]
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
errors.plot.bar(ax=ax, color=["#4c6e91","#297662","#ad592a"], rot=0)
ax.axhline(0,color="black",linewidth=.8)
ax.set(xlabel="", ylabel="Predicted minus manual rise (px)", title="Knee pass to contact")
fig.savefig(folder / "rise_errors.png", dpi=140)
plt.show()

## 같은 프레임에서 얻고 잃은 정탐

각 행은 동작 중 수동 박스가 있는 프레임입니다. 아래의 gained는 후보가 새로 맞힌 장면, lost는 기존 모드가 맞히고 후보가 놓친 장면입니다.

In [ ]:
hand = frames[frames.in_motion & frames.match.notna()]
matches = hand.pivot(index=["video_id","frame_index"], columns="mode", values="match")
for reference in ["mac_single","mac_cutmix"]:
    gained = matches.cosine_single & ~matches[reference]
    lost = ~matches.cosine_single & matches[reference]
    changes = matches[gained | lost].assign(change=gained[gained | lost].map({True:"gained",False:"lost"}))
    print(reference, "→ cosine_single")
    display(changes.reset_index())

## a03 가림과 b02 접촉 장면

연두 점선은 수동 원판 박스, 주황 실선은 각 모드가 최종 선택한 박스입니다. 제목에서 선택된 가중치와 IoU를 확인합니다.

In [ ]:
from matplotlib.patches import Rectangle
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb
samples = [("a03_sn_f_01",10),("b02_sn_h_02",36)]
fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
for row_axes, (video, index) in zip(axes,samples):
    rgb = read_rgb(VIDEOS / f"{video}.mp4", index)
    sample = frames[(frames.video_id==video) & (frames.frame_index==index)].set_index("mode")
    for ax, mode in zip(row_axes,summary.index):
        r = sample.loc[mode]
        x,y = r.gt_x1,r.gt_y1
        width,height = r.gt_x2-x,r.gt_y2-y
        ax.imshow(rgb)
        ax.add_patch(Rectangle((x,y),width,height,fill=False,edgecolor="lime",linestyle="--",lw=2))
        status = "missing"
        if pd.notna(r.x1):
            ax.add_patch(Rectangle((r.x1,r.y1),r.x2-r.x1,r.y2-r.y1,fill=False,edgecolor="darkorange",lw=2))
            status = f"{r.selected_model} | IoU {r.iou:.2f}"
        ax.set(xlim=(max(0,x-width*.5),min(rgb.shape[1],x+width*1.5)),
               ylim=(min(rgb.shape[0],y+height*1.5),max(0,y-height*.5)),
               title=f"{mode} | {video[:3]} #{index}\n{status}")
        ax.axis("off")
fig.savefig(folder / "frame_comparison_local.png", dpi=140)
plt.show()